# WEL01 · M08 대량 유입 시 경보 보존과 수신확인 점검 (학생 실습용)

- 본 노트북은 **미션8(M08) 전용 학습자 실습용** 노트북입니다.
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일을 업로드할 필요가 없습니다.
- 위에서부터 첫 실행 셀, 준비 셀, `STEP 01`~`05`의 `TODO`를 순서대로 완성한 뒤 마무리 셀을 실행하면 `WEL01/outputs/M08/WEL01_M08_result.html`이 생성됩니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='student'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M08 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M08"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M08 대량 유입 시 경보 보존과 수신확인 점검

평시 20명에서 120명까지 늘어난 부하와 통신 장애를 시험합니다. 응답 없는 경보를 완료 처리하면 실패가 감춰집니다. 생성 수, 수신확인 수, 미확인 보존 수를 따로 계산하고 영구 ACK 실패 상황까지 포함해 남은 확인 업무를 보고하세요.

실습 시간 210분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M08'
PARAMETERS = {'FULL_CAPACITY': 60}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M08 STEP 01 — 평시 인원 대비 증가율 계산하기

<h4>이 단계에서 해결할 일</h4><p>평시 20명에서 대규모 인원이 들어올 때 몇 배인지와 몇 퍼센트 증가했는지를 구분해야 시험 조건을 정확히 설명할 수 있습니다. 각 시험 상황의 인원 증가율을 계산합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m08_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>증가율_pct</code>이며 배정 시간은 42분입니다. 노트북의 <code>M08 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>병사 수</code> — 각 부하 시험 상황의 가상 인원입니다. 비교 기준은 모든 행에서 20명입니다.</li></ul><h4>수행 순서</h4><ol><li>raw의 7개 시험 상황과 병사 수를 확인합니다.</li><li>병사 수를 20으로 나눈 뒤 1을 빼고 100을 곱합니다.</li><li>결과를 증가율_pct 열에 넣어 step1에 저장합니다.</li><li>인원·기준 20명·증가율을 함께 적습니다. 배수도 쓸 경우 인원/20으로 별도 계산합니다.</li></ol><h4>결과를 읽는 방법</h4><p>120명은 평시의 6배이고 증가율은 500%입니다. 600% 증가로 쓰지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"병사 수": 20}</code></td><td><code>0</code></td><td>기준과 같으므로 증가하지 않았습니다.</td></tr><tr><td><code>{"병사 수": 120}</code></td><td><code>500</code></td><td>6배가 된 것은 500% 증가입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>상황마다 다른 기준 인원을 사용하지 않습니다. 인원 수가 없는 행은 20명으로 채우지 않습니다.</p>

In [ ]:
# M08 STEP 01 — 평시 인원 대비 증가율 계산하기
step1 = TODO(MID, 1, '평시 인원 대비 증가율 계산하기')
display(step1[["원행ID", '증가율_pct']].head(10))


### M08 STEP 02 — 생성된 경보가 기록에서 빠졌는지 계산하기

<h4>이 단계에서 해결할 일</h4><p>수신확인이 오지 않아도 미확인 경보가 보존되어 있으면 추적은 가능합니다. 생성된 경보가 수신확인과 미확인 보존 두 집합에 모두 빠짐없이 남는지 건수로 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m08_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>추적_유실수</code>이며 배정 시간은 42분입니다. 노트북의 <code>M08 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>생성 경보</code> — 시험 중 만들어진 전체 경보 수입니다.</li><li><code>수신확인</code> — ACK, 즉 수신확인 응답이 확인된 경보 수입니다.</li><li><code>미확인보존</code> — 응답은 없지만 다시 확인할 수 있도록 보존된 경보 수입니다.</li></ul><h4>수행 순서</h4><ol><li>step1에서 세 건수가 같은 시험 상황의 값인지 확인합니다.</li><li>생성 경보-수신확인-미확인보존을 계산합니다.</li><li>추적_유실수 열을 추가한 step2를 저장합니다. 음수도 그대로 남깁니다.</li><li>0이면 건수 합계가 맞고, 양수이면 기록에서 빠진 수가 있으며, 음수이면 중복 집계 등 합계 오류가 있을 수 있다고 기록합니다. 경보 ID의 중복 여부는 참고 로그에서 추가 확인합니다.</li></ol><h4>결과를 읽는 방법</h4><p>0은 경보가 모두 전달되었다는 뜻이 아니라 확인된 수와 보존된 수의 합이 생성 수와 같다는 뜻입니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"생성 경보": 100, "수신확인": 80, "미확인보존": 20}</code></td><td><code>0</code></td><td>건수 합계가 맞습니다.</td></tr><tr><td><code>{"생성 경보": 100, "수신확인": 80, "미확인보존": 15}</code></td><td><code>5</code></td><td>5건이 두 기록에 없습니다.</td></tr><tr><td><code>{"생성 경보": 100, "수신확인": 90, "미확인보존": 20}</code></td><td><code>-10</code></td><td>합계가 초과하여 확인이 필요합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>음수는 실제로 경보가 음수 개 유실되었다는 뜻이 아닙니다. 합계 불일치를 드러내는 값이므로 0으로 잘라 내지 않습니다.</p>

In [ ]:
# M08 STEP 02 — 생성된 경보가 기록에서 빠졌는지 계산하기
step2 = TODO(MID, 2, '생성된 경보가 기록에서 빠졌는지 계산하기')
display(step2[["원행ID", '추적_유실수']].head(10))


### M08 STEP 03 — 수신확인이 없는 경보 수 그대로 남기기

<h4>이 단계에서 해결할 일</h4><p>응답이 없다는 이유로 경보를 수신 완료 목록에 넣으면 실패 상황이 사라집니다. 현재 보존된 미확인 경보 수를 별도 열로 옮겨 다음 담당자가 확인할 잔량을 보여 줍니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m08_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>수신_미확인수</code>이며 배정 시간은 42분입니다. 노트북의 <code>M08 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>미확인보존</code> — ACK가 없어 완료로 처리하지 않은 채 남겨 둔 경보 수입니다.</li></ul><h4>수행 순서</h4><ol><li>step2의 미확인보존 값을 읽습니다.</li><li>값을 그대로 수신_미확인수 열에 복사하고 step3에 저장합니다.</li><li>영구 ACK 실패 시험에서는 생성 수 전부가 미확인으로 남는지 실제 로그를 확인합니다.</li><li>상황 이름·미확인 수·보존 위치를 기록합니다. 재시도 예정과 수신 완료를 구분합니다.</li></ol><h4>결과를 읽는 방법</h4><p>미확인 수가 남는 것은 감춰야 할 계산 오류가 아니라 시험이 드러낸 후속 확인 대상입니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"미확인보존": 20}</code></td><td><code>20</code></td><td>잔량을 그대로 유지합니다.</td></tr><tr><td><code>{"미확인보존": 0}</code></td><td><code>0</code></td><td>이 시험에서 남은 미확인이 없습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>수신확인 수를 늘려 잔량을 0으로 맞추지 않습니다. 보존은 전달 성공의 증거가 아닙니다.</p>

In [ ]:
# M08 STEP 03 — 수신확인이 없는 경보 수 그대로 남기기
step3 = TODO(MID, 3, '수신확인이 없는 경보 수 그대로 남기기')
display(step3[["원행ID", '수신_미확인수']].head(10))


### M08 STEP 04 — 미확인 경보와 신호두절을 각각 확인 대상으로 표시하기

<h4>이 단계에서 해결할 일</h4><p>전송 잔량이 없어도 신호가 끊긴 대상자는 확인이 필요할 수 있습니다. 수신확인 미완료와 신호두절 두 조건 중 하나라도 있으면 추가 확인 대상으로 표시합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m08_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>안전_확인필요</code>이며 배정 시간은 42분입니다. 노트북의 <code>M08 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>수신_미확인수</code> — STEP 3의 ACK 미확인 잔량입니다.</li><li><code>신호두절 경보</code> — 시험 중 생성된 두절 관련 경보 수입니다. 현재 전달 잔량과 다른 값입니다.</li></ul><h4>수행 순서</h4><ol><li>step3에서 미확인 수와 두절 경보 수를 읽습니다.</li><li>어느 한 값이라도 0보다 크면 안전_확인필요=True로 만듭니다.</li><li>둘 다 0이면 False로 하여 step4에 저장합니다.</li><li>True인 상황의 발생 사유를 각각 적습니다. 두절 경보가 누적 발생 수인지 현재 미해결 수인지 자료에서 확인 가능한 범위를 명시합니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 이 시험에서 추가 확인할 사유가 있다는 뜻입니다. False는 모든 운영 위험이 없다는 보증이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"수신_미확인수": 0, "신호두절 경보": 1}</code></td><td><code>True</code></td><td>전달 잔량이 없어도 두절 확인이 필요합니다.</td></tr><tr><td><code>{"수신_미확인수": 1, "신호두절 경보": 0}</code></td><td><code>True</code></td><td>미확인이 남아 있습니다.</td></tr><tr><td><code>{"수신_미확인수": 0, "신호두절 경보": 0}</code></td><td><code>False</code></td><td>두 조건의 발생 수가 모두 0입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>이 열은 추적 건수 불일치를 검사하지 않습니다. 추적_유실수는 다음 단계에서 별도로 반영합니다.</p>

In [ ]:
# M08 STEP 04 — 미확인 경보와 신호두절을 각각 확인 대상으로 표시하기
step4 = TODO(MID, 4, '미확인 경보와 신호두절을 각각 확인 대상으로 표시하기')
display(step4[["원행ID", '안전_확인필요']].head(10))


### M08 STEP 05 — 추적 오류와 수신 미확인을 구분해 결과 적기

<h4>이 단계에서 해결할 일</h4><p>장애 시험 결과를 성공·실패 하나로만 적으면 아직 남은 일이 사라집니다. 건수 불일치를 먼저 확인하고, 합계가 맞더라도 응답이 없으면 미확인 보존 상태로 보고합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m08_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>전달_판정</code>이며 배정 시간은 42분입니다. 노트북의 <code>M08 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>추적_유실수</code> — STEP 2의 건수 차이입니다. 0 이외의 값은 모두 합계 확인 대상입니다.</li><li><code>수신_미확인수</code> — STEP 3의 미확인 잔량입니다.</li></ul><h4>수행 순서</h4><ol><li>step4에서 추적_유실수가 0인지 먼저 확인합니다.</li><li>0이 아니면 추적 유실 보완으로 표시합니다. 합계가 맞고 미확인 수가 1 이상이면 미확인 보존·담당 인계로 표시합니다.</li><li>합계가 맞고 미확인 수가 0일 때만 현재 시험 수신확인 완료로 표시하여 step5에 저장합니다.</li><li>미확인 또는 합계 오류 상황은 원행ID·건수·담당 역할·재확인 시점을 기록합니다. 안전_확인필요도 함께 보여 두절 확인을 빠뜨리지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>수신확인 완료라는 문구는 해당 시험의 전달 기록에 한정됩니다. 두절 대상 확인이나 전체 업무 완료까지 뜻하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"추적_유실수": -1, "수신_미확인수": 0}</code></td><td><code>추적 유실 보완</code></td><td>음수도 불일치입니다.</td></tr><tr><td><code>{"추적_유실수": 0, "수신_미확인수": 3}</code></td><td><code>미확인 보존·담당 인계</code></td><td>합계는 맞지만 응답이 없습니다.</td></tr><tr><td><code>{"추적_유실수": 0, "수신_미확인수": 0}</code></td><td><code>현재 시험 수신확인 완료</code></td><td>이 시험의 전달 확인이 끝났습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>영구 ACK 실패 시험은 추적 유실 0이어도 미확인 보존·담당 인계로 남아야 합니다. 전송 지연·두절 탐지 시간은 별도 실제 로그로 평가합니다.</p>

In [ ]:
# M08 STEP 05 — 추적 오류와 수신 미확인을 구분해 결과 적기
step5 = TODO(MID, 5, '추적 오류와 수신 미확인을 구분해 결과 적기')
display(step5[["원행ID", '전달_판정']].head(10))


### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
